# Compaction using OPTIMIZE and Z-Ordering

### OPTIMIZE Command

> Merges multiple small files into fewer, larger files, thus improves the performance

#### 1. Create table - demo.delta_lake.optimize_stock_prices

In [0]:
DROP TABLE IF EXISTS demo.delta_lake.optimize_stock_prices;
CREATE TABLE IF NOT EXISTS demo.delta_lake.optimize_stock_prices
(
  stock_id STRING,
  price DOUBLE,
  trading_date DATE
);

#### 2. Inserts some data in a few transactions

In [0]:
INSERT INTO demo.delta_lake.optimize_stock_prices
VALUES ('AAPL', 227.65, "2025-02-10");

num_affected_rows,num_inserted_rows
1,1


In [0]:
INSERT INTO demo.delta_lake.optimize_stock_prices
VALUES ('GOOGL', 2775.0, "2025-02-10");

num_affected_rows,num_inserted_rows
1,1


In [0]:
INSERT INTO demo.delta_lake.optimize_stock_prices
VALUES ('MSFT', 325.0, "2025-02-10");

num_affected_rows,num_inserted_rows
1,1


In [0]:
INSERT INTO demo.delta_lake.optimize_stock_prices
VALUES ('AMZN', 3520.0, "2025-02-12");

num_affected_rows,num_inserted_rows
1,1


#### 3. Check the table history

In [0]:
DESC HISTORY demo.delta_lake.optimize_stock_prices;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
5,2025-02-20T13:39:17Z,3341927249304550,cloudboxacademy@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> false, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(74520196940261),0122-185524-t8czgn0m,4,SnapshotIsolation,false,"Map(numRemovedFiles -> 4, numRemovedBytes -> 4643, p25FileSize -> 1232, numDeletionVectorsRemoved -> 0, minFileSize -> 1232, numAddedFiles -> 1, maxFileSize -> 1232, p75FileSize -> 1232, p50FileSize -> 1232, numAddedBytes -> 1232)",null,Databricks-Runtime/15.4.x-scala2.12
4,2025-02-20T13:36:17Z,3341927249304550,cloudboxacademy@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(74520196940261),0122-185524-t8czgn0m,3,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1159)",null,Databricks-Runtime/15.4.x-scala2.12
3,2025-02-20T13:36:14Z,3341927249304550,cloudboxacademy@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(74520196940261),0122-185524-t8czgn0m,2,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1159)",null,Databricks-Runtime/15.4.x-scala2.12
2,2025-02-20T13:36:12Z,3341927249304550,cloudboxacademy@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(74520196940261),0122-185524-t8czgn0m,1,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1166)",null,Databricks-Runtime/15.4.x-scala2.12
1,2025-02-20T13:36:07Z,3341927249304550,cloudboxacademy@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(74520196940261),0122-185524-t8czgn0m,0,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1159)",null,Databricks-Runtime/15.4.x-scala2.12
0,2025-02-20T13:35:33Z,3341927249304550,cloudboxacademy@gmail.com,CREATE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.enableDeletionVectors"":""true""}, statsOnLoad -> false)",null,List(74520196940261),0122-185524-t8czgn0m,null,WriteSerializable,true,Map(),null,Databricks-Runtime/15.4.x-scala2.12


#### 4. Check the number of files required to get the latest data

In [0]:
DESC DETAIL demo.delta_lake.optimize_stock_prices;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics
delta,219e769d-bbd8-4f81-a25e-e581aa07a513,demo.delta_lake.optimize_stock_prices,null,abfss://demo@deacourseextdl.dfs.core.windows.net/delta_lake/__unitystorage/schemas/3a4b6eb6-1de9-4154-9932-f9ba58518ec2/tables/992bc91c-9547-4922-b289-906f43ead285,2025-02-20T13:35:32.371Z,2025-02-20T13:36:17Z,List(),List(),4,4643,Map(delta.enableDeletionVectors -> true),3,7,List(deletionVectors),"Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)"


In [0]:
SELECT * FROM demo.delta_lake.optimize_stock_prices VERSION AS OF 3;

stock_id,price,trading_date
GOOGL,2775.0,2025-02-10
AAPL,227.65,2025-02-10
MSFT,325.0,2025-02-10


#### 5. Run OPTIMIZE

In [0]:
OPTIMIZE demo.delta_lake.optimize_stock_prices;

path,metrics
abfss://demo@deacourseextdl.dfs.core.windows.net/delta_lake/__unitystorage/schemas/3a4b6eb6-1de9-4154-9932-f9ba58518ec2/tables/992bc91c-9547-4922-b289-906f43ead285,"List(1, 4, List(1232, 1232, 1232.0, 1, 1232), List(1159, 1166, 1160.75, 4, 4643), 0, null, null, 0, 1, 4, 0, true, 0, 0, 1740058754363, 1740058759325, 4, 1, null, List(0, 0), 3, 3, 265, 0, null)"


In [0]:
OPTIMIZE demo.delta_lake.optimize_stock_prices
ZORDER BY stock_id;

path,metrics
abfss://demo@deacourseextdl.dfs.core.windows.net/delta_lake/__unitystorage/schemas/3a4b6eb6-1de9-4154-9932-f9ba58518ec2/tables/992bc91c-9547-4922-b289-906f43ead285,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, List(minCubeSize(107374182400), List(0, 0), List(1, 1232), 0, List(0, 0), 0, null), null, 0, 0, 1, 1, false, 0, 0, 1740058993738, 1740058995523, 4, 0, null, List(0, 0), 3, 3, 0, 0, null)"
